# 00 — Setup and ingest

**One-time setup notebook.** Run this once per BEIR dataset you want
to work with. The remaining lab notebooks (`01`, `02`, `03`) read
from the collections this notebook builds.

## What this notebook does

1. Checks your `.env` for the credentials the lab needs.
2. Lets you pick one (or more) of 8 BEIR retrieval datasets.
3. Ingests the dataset into MongoDB Atlas:
   - Splits documents into chunks.
   - Embeds the chunks with `voyage-context-3` (a *contextualized*
     chunk embedder — each chunk's vector is informed by the whole
     document it came from).
   - Writes chunks + vectors to a collection.
   - Builds both a **vector** search index and a **lexical (BM25)**
     search index so we can compare retrieval strategies later.

Once a dataset is ingested, you can re-run the other notebooks
against it as many times as you want without re-ingesting.

> **Cost / time:** Each ingest at the default sample of 500
> documents takes ~30–90 s and uses a few cents of Voyage credit.

## Prerequisites

Add the following to a file named `.env` at the repo root:

```
VOYAGE_API_KEY=al-...        # from Atlas → AI Models → API Keys
MONGODB_URI=mongodb+srv://...
OPENAI_API_KEY=sk-...        # only needed in notebook 03
```

Then install the Python deps from the repo root:

```
pip3 install --break-system-packages \
    requests pymongo voyageai beir python-dotenv \
    tqdm openai matplotlib pandas
```

In [ ]:
import os, sys
# Notebooks live in agent-notebooks/. Add the repo root to the path so
# we can import the lab's library modules.
_REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

In [ ]:
# Confirm the credentials are loaded.
import lib
lib.require_credentials()
print("VOYAGE_API_KEY loaded:", bool(lib.VOYAGE_API_KEY))
print("MONGODB_URI loaded:   ", bool(lib.MONGODB_URI))

## Pick a dataset

BEIR (Benchmarking IR) is a collection of retrieval datasets with
human-curated relevance judgements (qrels). Each entry below is a
full `(corpus, queries, qrels)` triple:

| name | what it tests |
|---|---|
| `scifact` | scientific claim verification (300 queries / 5.2k abstracts) |
| `nfcorpus` | medical literature retrieval (323 queries / 3.6k docs) |
| `fiqa` | financial Q&A — opinionated long answers (648 queries / 57k docs) |
| `arguana` | counter-argument retrieval (1.4k queries / 8.7k arguments) |
| `scidocs` | scientific paper retrieval (1k queries / 25k docs) |
| `trec-covid` | COVID-19 research retrieval (50 queries / 171k docs) |
| `touche2020` | controversial-topic argument retrieval (49 queries / 382k docs) |
| `quora` | duplicate-question retrieval (10k queries / 523k docs) |

**Recommendation for a first run: `scifact`.** It's small, the
queries are short scientific claims, and the qrels are clean. Once
you've worked through the lab on `scifact`, try a different domain
(`fiqa` for financial, `nfcorpus` for medical) and watch how
metrics change.

In [ ]:
# ── EDIT ME ─────────────────────────────────────────────────────
DATASET       = "scifact"   # one of the names above
CORPUS_SAMPLE = 500          # how many docs to ingest
# ────────────────────────────────────────────────────────────────

from lib import DATASETS
assert DATASET in DATASETS, f"Unknown dataset: {DATASET}"
print(f"Will ingest {DATASET}: {DATASETS[DATASET]['description']}")

## Ingest

This runs the full pipeline: download → chunk → embed → write to
MongoDB → build both indexes. A status bar in stdout tracks
progress. Indexes take 30–60 s to become queryable after writes
finish; the script waits for that automatically.

Re-running this cell **drops** the dataset's collection and
re-ingests from scratch.

In [ ]:
from ingest import ingest
ingest(DATASET, corpus_sample=CORPUS_SAMPLE)

## Verify

Sanity-check that the collection and both indexes exist and are
queryable. If the text index is still building, the next notebook
will fail with an `index not found` error; wait a minute and
re-run the verification cell.

In [ ]:
import pymongo
from lib import MONGODB_URI, DB_NAME, INDEX_NAME, collection_name
from retrieve import TEXT_INDEX_NAME

client = pymongo.MongoClient(MONGODB_URI)
coll   = client[DB_NAME][collection_name(DATASET)]

print(f"Collection: {coll.full_name}")
print(f"  chunks  : {coll.estimated_document_count():,}")
print()
print(f"Search indexes:")
for idx in coll.list_search_indexes():
    print(f"  {idx['name']:<25}  queryable={idx.get('queryable', False)}")
client.close()

## Next

Open **`01_evaluate_blackbox.ipynb`** to see how IR evaluation
works — we'll treat lexical (BM25) retrieval as a "black box" and
measure its quality with the same metrics you'd use to compare
any retrieval system.